# 🎤 KaraokeAI Backend Server
### Run this notebook ONCE. Future updates auto-apply via git pull!
**Steps:** Click **Runtime → Run all** after setting your ngrok token in Cell 3.

In [ ]:
# ── CELL 1: Install all dependencies ──────────────────────────
print('Installing dependencies... (takes 3-4 minutes)')

# Core karaoke + API packages
!pip install -q faster-whisper demucs ffmpeg-python pysubs2 fastapi uvicorn pyngrok nest-asyncio transliterate anyascii aksharamukha 'fastapi[standard]'

# System dependencies for aeneas
!apt-get install -q -y espeak espeak-data libespeak-dev ffmpeg

# Install aeneas (forced alignment library)
!pip install -q numpy
!pip install -q aeneas

# Verify aeneas works
import subprocess
r = subprocess.run(['python', '-m', 'aeneas.diagnostics'], capture_output=True, text=True)
print('aeneas diagnostics:', r.stdout[-500:] if r.stdout else r.stderr[-300:])

print('✅ All dependencies installed!')

In [ ]:
# ── CELL 2: Clone/update your GitHub repository ───────────────
# api_server.py lives in the repo now — git pull gets the latest automatically!
import os

GITHUB_REPO = 'https://github.com/shanmuganathan457/karaoke-video-generator.git'

if os.path.exists('karaoke-video-generator'):
    print('Repo already exists, pulling latest changes...')
    os.system('cd karaoke-video-generator && git pull')
else:
    print('Cloning your GitHub repository...')
    os.system(f'git clone {GITHUB_REPO}')

os.chdir('karaoke-video-generator')
print(f'✅ Working directory: {os.getcwd()}')
print('📁 api_server.py exists:', os.path.exists('api_server.py'))

In [ ]:
# ── CELL 3: Enter your ngrok token ────────────────────────────
# Get your FREE token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set!')

In [ ]:
# ── CELL 4: START THE SERVER ──────────────────────────────────
import subprocess, time, sys
from pyngrok import ngrok

ngrok.kill()

# Start uvicorn — logs to file so we can debug if it crashes
log = open('/tmp/uvicorn.log', 'w')
proc = subprocess.Popen(
    ['uvicorn', 'api_server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log, stderr=log
)

time.sleep(5)

if proc.poll() is not None:
    print('❌ Server CRASHED! Error:')
    print(open('/tmp/uvicorn.log').read())
    raise RuntimeError('uvicorn failed to start. See error above.')

public_url = ngrok.connect(8000)

print('=' * 60)
print('🚀 KaraokeAI Backend is LIVE!')
print('=' * 60)
print(f'\n📡 YOUR PUBLIC API URL:')
print(f'   {public_url}')
print(f'\n✅ Copy this URL and give it to your developer!')
print(f'⚠️  Keep this tab open — closing it stops the server.')
print('=' * 60)

while True:
    time.sleep(60)
    lines = open('/tmp/uvicorn.log').readlines()
    last = lines[-1].strip() if lines else '...'
    print(f'💓 Server still running...')